# Раздел 11 · Интеграции: LangChain, LlamaIndex, n8n

> Этот ноутбук продолжает Раздел 3 («Коллекции и точки») — использует уже созданную там коллекцию `BOOKS_COLLECTION`. Если ниже вылетит `AssertionError: Коллекция ... ещё не создана` — сначала откройте и выполните ноутбук Раздела 3, потом возвращайтесь сюда.

In [1]:
%pip install qdrant-client sentence-transformers ipykernel langchain-qdrant langchain-core --quiet


/home/dimon/vscode/examples_qdrant/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


> Если сразу после установки следующая ячейка падает с `ModuleNotFoundError` на только что установленном пакете — перезапустите ядро (Kernel → Restart) и выполните ячейки заново с начала: иногда ядро не видит пакет, установленный в уже запущенном процессе, без перезапуска.

In [2]:
import json
import subprocess
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
)

print("OK: библиотеки импортированы")


/home/dimon/vscode/examples_qdrant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK: библиотеки импортированы


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [3]:
CONTAINER_NAME = "qdrant_stepik"



def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6333 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)
client = QdrantClient(
    url="http://localhost:6336",
    api_key="Fucking_password_1234567890",
)
print(client.get_collections())


qdrant_stepik
collections=[CollectionDescription(name='day1_semantic_search'), CollectionDescription(name='exercise1_books')]


/tmp/ipykernel_183827/1723463311.py:21: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


In [4]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35017.05it/s]


Размерность эмбеддинга: 384 | книг в подборке: 87


In [5]:
assert client.collection_exists(BOOKS_COLLECTION), (
    "Коллекция {} ещё не создана. Откройте и выполните ноутбук "
    "Раздела 3 (\"Коллекции и точки\") первым - там она создаётся и наполняется."
).format(BOOKS_COLLECTION)
print("OK:", BOOKS_COLLECTION, "-", client.count(BOOKS_COLLECTION).count, "точек")


OK: exercise1_books - 87 точек


## Интеграции: LangChain

Подключаем LangChain к уже существующей `BOOKS_COLLECTION` через `from_existing_collection` —
никакой новой коллекции, никакого API-ключа LLM-провайдера не требуется вообще: нужен только
retrieval (поиск), а не генерация ответа, поэтому достаточно обернуть уже готовую
`sentence-transformers`-модель в интерфейс `Embeddings` из LangChain.

### Пример — from_existing_collection и неочевидная особенность фреймворков

In [6]:
from langchain_core.embeddings import Embeddings
from langchain_qdrant import QdrantVectorStore


class E5Embeddings(Embeddings):
    # адаптер: LangChain ожидает интерфейс Embeddings, а не сырую sentence-transformers модель -
    # префиксы "passage: "/"query: " остаются нашей ответственностью, фреймворк о них не знает
    def embed_documents(self, texts):
        return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()

    def embed_query(self, text):
        return model.encode("query: " + text, normalize_embeddings=True).tolist()


vector_store = QdrantVectorStore.from_existing_collection(
    embedding=E5Embeddings(),
    collection_name=BOOKS_COLLECTION,
    url="http://localhost:6336",
    api_key="Fucking_password_1234567890",
    content_payload_key="text",  # у нас нет поля "page_content", есть "text"
)

docs = vector_store.similarity_search("мрачная история об убийстве и мучительных муках совести", k=3)
for doc in docs:
    print("page_content:", doc.page_content[:60], "...")
    print("metadata:    ", doc.metadata)
    print()


page_content: Бедный студент Родион Раскольников, одержимый теорией о прав ...
metadata:     {'_id': 10, '_collection_name': 'exercise1_books'}

page_content: Попутчик в поезде, дворянин Позднышев, рассказывает случайно ...
metadata:     {'_id': 32, '_collection_name': 'exercise1_books'}

page_content: Бывший монах, а ныне послушник Иван Флягин рассказывает попу ...
metadata:     {'_id': 41, '_collection_name': 'exercise1_books'}



/home/dimon/vscode/examples_qdrant/.venv/lib/python3.14/site-packages/langchain_qdrant/qdrant.py:465: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


Реальный результат: `page_content` подтягивается верно (топ-3 — id 10, 32, 41, тот же,
что и при поиске через сырой клиент), а вот `metadata` содержит только `{'_id': ..., '_collection_name': ...}` — **пустая**
в остальном. Дело не в баге, а в соглашении LangChain: по умолчанию оно ищет вложенный ключ
`"metadata"` внутри payload (`metadata_payload_key="metadata"`), а у нас payload плоский —
поля `genre`, `year`, `author` лежат прямо в корне, а не под ключом `metadata`. Framework
не изобретает свою модель данных, но и не читает мысли — если коллекция построена вручную
(как весь этот ноутбук), а не через `from_documents`, забирать остальной payload приходится
отдельно, через `doc.metadata["_id"]` и сырой клиент.

## Задание 1

По результатам поиска выше (`docs`) для каждого документа получите полный payload (включая
`genre`, `author`, `year`) через `client.retrieve(BOOKS_COLLECTION, ids=[doc.metadata["_id"] for doc in docs])`.
Сохраните в `full_payloads` и убедитесь, что у каждого совпадает `title`, который LangChain
не увидел бы через `doc.metadata`, но реально хранится в коллекции.

**Используйте:** `client.retrieve(...)`.

In [7]:
# ВАШ КОД ЗДЕСЬ
full_payloads = client.retrieve(
    collection_name=BOOKS_COLLECTION,
    ids=[doc.metadata["_id"] for doc in docs],
)


In [8]:
def check_ex1(docs, full_payloads):
    assert len(full_payloads) == len(docs), "количество не совпадает"
    ids_from_docs = {doc.metadata["_id"] for doc in docs}
    ids_from_payloads = {p.id for p in full_payloads}
    assert ids_from_docs == ids_from_payloads, "id не совпадают между docs и full_payloads"
    for p in full_payloads:
        assert "genre" in p.payload and "author" in p.payload and "year" in p.payload, (
            "полный payload точки id={} должен содержать genre, author и year".format(p.id)
        )
    print("OK:", [(p.id, p.payload["genre"], p.payload["author"], p.payload["year"]) for p in full_payloads])
    print("ОТВЕТ ДЛЯ STEPIK:", full_payloads[0].payload["year"])


check_ex1(docs, full_payloads)


OK: [(10, 'роман', 'Фёдор Достоевский', 1866), (32, 'повесть', 'Лев Толстой', 1889), (41, 'повесть', 'Николай Лесков', 1873)]
ОТВЕТ ДЛЯ STEPIK: 1866


### Пример — как это выглядит "из коробки": from_documents

Пустой `doc.metadata` выше — не ограничение LangChain, а следствие того, что `BOOKS_COLLECTION`
собрана в обход фреймворка, сырым `client.upsert`. Если строить коллекцию через сам LangChain
(`from_documents`), никакого воркэраунда не нужно — фреймворк сам раскладывает `page_content`/
`metadata` по payload при записи и сам же читает их обратно в том же формате:

In [ ]:
from langchain_core.documents import Document

LANGCHAIN_DEMO_COLLECTION = PREFIX + "books_langchain_demo"

demo_books = [BOOK_BY_ID[10], BOOK_BY_ID[32], BOOK_BY_ID[41]]  # те же три книги, что в топ-3 выше
demo_docs = [
    Document(
        page_content=b["text"],
        metadata={"title": b["title"], "genre": b["genre"], "author": b["author"], "year": b["year"]},
    )
    for b in demo_books
]

if client.collection_exists(LANGCHAIN_DEMO_COLLECTION):
    client.delete_collection(LANGCHAIN_DEMO_COLLECTION)

# from_documents сам создаёт коллекцию под размерность эмбеддинга и сам раскладывает payload -
# никаких content_payload_key/metadata_payload_key указывать не нужно, это как раз дефолты
demo_store = QdrantVectorStore.from_documents(
    demo_docs,
    embedding=E5Embeddings(),
    url="http://localhost:6333",
    collection_name=LANGCHAIN_DEMO_COLLECTION,
)

demo_hits = demo_store.similarity_search("мрачная история об убийстве и мучительных муках совести", k=1)
for doc in demo_hits:
    print("page_content:", doc.page_content[:60], "...")
    print("metadata:    ", doc.metadata)


Реальный результат: `metadata` теперь содержит `{'title': 'Преступление и наказание', 'genre': 'роман', 'author': 'Фёдор Достоевский', 'year': 1866, '_id': ..., '_collection_name': ...}` —
всё на месте, без единого обращения к сырому клиенту. Разница с примером выше не в версии
LangChain и не в баге, а исключительно в том, кто строил коллекцию: сам фреймворк (тогда он
помнит свою же раскладку payload) или внешний код (тогда фреймворк видит только то, что вы ему
явно показали через `content_payload_key`).